# scqubits example: when does parallelizing a sweep help? (`num_cpus`, BLAS threads)

J. Koch and P. Groszkowski

For further documentation of scqubits see https://scqubits.readthedocs.io/en/latest/.

---

`ParameterSweep` can compute a sweep across worker processes via its `num_cpus`
argument. **Parallelism is not free, and on small sweeps it does *not* help — sometimes
it makes them slower.** Whether it pays off is governed by one simple balance:

> parallelism helps only when&nbsp;&nbsp;**(number of grid points) × (cost per point)&nbsp;&nbsp;≫&nbsp;&nbsp;the fixed overhead**
> of starting worker processes and shipping each task to them.

So the behaviour you should *expect*:

- **Few grid points:** `num_cpus > 1` gives no speedup, and can be *slower* than serial.
  This is normal — keep the default `num_cpus = 1`.
- **Many grid points** (and/or an expensive-per-point system): workers pay off.

A second knob interacts with the first: every eigensolve runs on a multithreaded
**BLAS/LAPACK** backend. If you run several workers and let each use all cores, you
oversubscribe — which on large dense matrices is not a small slowdown but a
**catastrophe** (~90× in the example below). So we cap BLAS threads **first**, before
importing anything. Timings are machine-specific — run the cells yourself; the
illustrative numbers are from a 10-core Mac mini and yours will differ.

## Step 0 (do this first): cap BLAS threads *before* importing scqubits

The BLAS backend reads its thread count **once, at import time**, so this must be the
**first cell you run in a fresh kernel** — before `numpy` or `scqubits` is imported. (If
scqubits is already imported, restart the kernel and run this first.) Capping to `1` is a
good starting point for qubit sweeps and prevents oversubscription once `num_cpus > 1`;
the 'BLAS footgun' section below explains why this matters so much.


In [ ]:
import os

# MUST run before numpy / scqubits are imported (restart the kernel otherwise).
for _var in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "OMP_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_var] = "1"


### Confirm the cap took effect

The single most common cause of confusing sweep timings is that the cap **did not apply**
— because `numpy`/`scqubits` were already imported when the env vars were set. Check it
(needs `threadpoolctl`, `pip install threadpoolctl`); each reported count should be `1`:


In [ ]:
import numpy as np
import scqubits as scq  # imports scipy, whose BLAS the eigensolvers use

try:
    from threadpoolctl import threadpool_info

    pools = threadpool_info()
    for pool in pools:
        print(f"{pool['internal_api']:>10}: {pool['num_threads']} thread(s)")
    if not pools:
        print("(no controllable BLAS pool detected)")
    print("\nEach count should be 1. If not, restart the kernel and run the cap cell")
    print("FIRST. (On Apple Silicon, numpy itself uses Accelerate, which ignores these")
    print(" variables; scipy's bundled OpenBLAS -- shown here -- is what scqubits uses.)")
except ImportError:
    print("threadpoolctl not installed (pip install threadpoolctl) -- skipping check.")


## A small timing helper and an example system

Three coupled tunable transmons, swept over the flux of the first. We rebuild a fresh
sweep for each timed run, and we will sweep **the same system at two grid sizes** to see
both regimes. Wall-clock timing is noisy, so we take the median of a few repeats and
discard a warm-up run (the first parallel run pays a one-time process-startup cost).


In [ ]:
import time


def make_sweep(num_cpus, n_points, truncated_dim=6):
    qubits = [
        scq.TunableTransmon(
            EJmax=30.0, EC=0.2, d=0.1, flux=0.0, ng=0.0, ncut=50,
            truncated_dim=truncated_dim, id_str=f"tmon{i}",
        )
        for i in range(3)
    ]
    hs = scq.HilbertSpace(qubits)
    for i in range(2):
        hs.add_interaction(g_strength=0.1, op1=qubits[i].n_operator, op2=qubits[i + 1].n_operator)
    flux_vals = np.linspace(0.0, 0.5, n_points)

    def update(flux):
        qubits[0].flux = flux

    return scq.ParameterSweep(
        hilbertspace=hs, paramvals_by_name={"flux": flux_vals},
        update_hilbertspace=update, evals_count=20, num_cpus=num_cpus, autorun=True,
    )


def time_run(n_points, num_cpus, repeats=3):
    """Median wall time of make_sweep(num_cpus, n_points), after a discarded warm-up."""
    make_sweep(num_cpus, n_points)  # warm-up (discarded)
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        make_sweep(num_cpus, n_points)
        times.append(time.perf_counter() - start)
    return float(np.median(times))


cores = os.cpu_count() or 1


### Running this as a `.py` script? Guard the entry point

These cells run fine **in Jupyter**. But if you move parallel code (`num_cpus > 1`) into a
plain Python script, note that macOS and Windows start workers with the `spawn` method,
which **re-imports your script** in each worker — so the entry point must be guarded, or
Python raises a `RuntimeError`:

```python
if __name__ == "__main__":
    ...   # code that triggers num_cpus > 1
```

Linux (which uses `fork`) and Jupyter need no guard. scqubits also prints a one-time
reminder the first time it spawns workers outside Jupyter.

## Demo 1 — a small sweep: parallelism does *not* help

Only **16 grid points**. There are too few points to amortize the cost of starting
workers and shipping tasks, so spreading them across processes buys nothing.


In [ ]:
for n in [c for c in (1, 2, 4) if c <= cores]:
    print(f"num_cpus={n}:  {time_run(16, n):.3f} s (median)")


**What to expect:** roughly *flat*, and `num_cpus = 2`/`4` may be slightly **slower** than
`1`. On the reference 10-core Mac mini (16 points, BLAS capped to 1):

| num_cpus | wall time | speedup |
|---:|---:|---:|
| 1 | 0.10 s | 1.00× |
| 2 | 0.10 s | 0.93× |
| 4 | 0.11 s | 0.84× |

This is the **normal** outcome for a quick sweep, and **not a bug** — with only 16 points
the per-task overhead dominates, so `num_cpus = 1` (the default) is the right choice.

## Demo 2 — a large sweep: parallelism pays off

The *same system*, now with **384 grid points**. There is enough work to amortize the
fixed overhead, so the workers help.


In [ ]:
for n in [c for c in (1, 2, 4) if c <= cores]:
    print(f"num_cpus={n}:  {time_run(384, n):.3f} s (median)")


**What to expect:** now the workers pay off. On the reference Mac mini (384 points, BLAS
capped to 1):

| num_cpus | wall time | speedup |
|---:|---:|---:|
| 1 | 2.14 s | 1.00× |
| 2 | 1.50 s | 1.42× |
| 4 | 0.90 s | **2.37×** |

Nothing changed except the **number of grid points** (16 → 384). That is the lever:
parallelism helps once *grid size × per-point cost* clears the overhead. The crossover
depends on the per-point cost too — a heavier system (larger `truncated_dim`, or a large
composite Hilbert space) reaches it at fewer points; a cheaper one needs more.

## The BLAS-thread footgun — *why* we capped first

Each eigensolve runs on a multithreaded BLAS backend that by default uses *all* cores.
Run `num_cpus` workers and each launches a full BLAS pool, so you oversubscribe the cores
by a factor of `num_cpus`. On small matrices this is wasteful; on **large dense** matrices
it is *catastrophic*. Measured on the reference Mac mini — 5 capacitively coupled fluxonia,
dressed dim 3125, dense diagonalization, 16-point sweep:

| configuration | wall time |
|---|---:|
| `num_cpus=1` | 42 s |
| `num_cpus=4`, BLAS **uncapped** | **3608 s**  (≈ 90× *slower*) |
| `num_cpus=4`, BLAS capped to 1 | 40 s |
| `num_cpus=8`, BLAS capped to 1 | 28 s |

40 threads fighting over 10 cores on dense LAPACK collapses performance — the cap is the
difference between working and broken. (Skip Step 0 and rerun Demo 2 with the cap off and
you will reproduce a milder version: `num_cpus = 2`/`4` become *much* slower than `1`.)
That is also the usual reason a `num_cpus` comparison looks 'inconclusive' or backwards:
the cap was never in effect.

**Caveat — `1` is not universally optimal.** On small matrices a single BLAS thread is
best; on large dense matrices each eigensolve benefits from several threads, so the
fastest setting balances the knobs: roughly **`num_cpus × BLAS-threads ≈ cores`**. The
optimum depends on machine, BLAS library, and matrix size — measure it.

The thread cap above was set with environment variables (the reliable way, since BLAS
reads them at import). scqubits also exposes the relevant knobs as **settings** you can
change at any time in a session:


In [ ]:
# Global default number of worker processes (used when `num_cpus` is not passed):
scq.settings.NUM_CPUS = 1

# Cap BLAS threads per worker process during parallel sweeps. This is applied only while
# the worker pool is created. Spawn-based workers (macOS, Windows) re-read the env vars at
# import; fork-based workers (Linux) need `threadpoolctl`. Keep num_cpus x BLAS-threads ~ cores.
scq.settings.MULTIPROC_BLAS_THREADS = 1

## For large composite systems, try sparse diagonalization first

Before reaching for `num_cpus`, note that for large coupled systems the dominant cost is
the *per-point diagonalization*, usually a bigger lever than parallelism. Recent scqubits
automatically uses **sparse** diagonalization (`scipy.eigsh`) for the default method when
only a few eigenstates of a large Hilbert space are requested
(`scqubits.settings.AUTO_SPARSE_DIAG`). For the 5-fluxonia system above (dressed dim
3125) this alone is ~16× faster per point than dense — far more than parallelism buys
there. And once sparse makes each point cheap, `num_cpus > 1` helps even less. **Try
sparse first; parallelize second.**


The setting is `scqubits.settings.AUTO_SPARSE_DIAG` (on by default). Here it is on a
moderately large system you can actually run — **4 capacitively coupled fluxonia**,
dressed dimension 6⁴ = 1296 — timing the same 8-point sweep with sparse vs dense
diagonalization:


In [ ]:
import warnings

def make_fluxonia_sweep(n_points=8):
    fluxonia = [
        scq.Fluxonium(EJ=4.0, EC=1.0, EL=1.0, flux=0.5, cutoff=110, truncated_dim=6, id_str=f"flx{i}")
        for i in range(4)
    ]
    hs = scq.HilbertSpace(fluxonia)
    for i in range(3):
        hs.add_interaction(g_strength=0.1, op1=fluxonia[i].n_operator, op2=fluxonia[i + 1].n_operator)

    def update(flux):
        fluxonia[0].flux = flux

    return scq.ParameterSweep(
        hilbertspace=hs, paramvals_by_name={"flux": np.linspace(0.45, 0.55, n_points)},
        update_hilbertspace=update, evals_count=20, num_cpus=1, autorun=True,
    )


def time_fluxonia():
    make_fluxonia_sweep()  # warm-up
    start = time.perf_counter()
    make_fluxonia_sweep()
    return time.perf_counter() - start


scq.settings.AUTO_SPARSE_DIAG = True
t_sparse = time_fluxonia()

# The dense path exercises scipy's matrix-function code, which emits spurious 'matmul'
# RuntimeWarnings on some BLAS backends (e.g. Apple Accelerate); silence them for the demo.
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    scq.settings.AUTO_SPARSE_DIAG = False
    t_dense = time_fluxonia()
scq.settings.AUTO_SPARSE_DIAG = True  # restore the default

print(f"sparse (default): {t_sparse:.2f} s    dense: {t_dense:.2f} s    speedup: {t_dense / t_sparse:.1f}x")


On the reference Mac mini this prints roughly `sparse 0.55 s   dense 3.94 s   speedup 7.2x`.
The advantage grows with system size, and for very large Hilbert spaces dense may not fit
in memory at all. (The dense run may also emit a few scipy-internal `matmul` warnings —
another reason to prefer the sparse default.)

## Letting scqubits pick the settings

You don't have to find the break-even by hand. `scqubits.recommend_parallelization` reads
the workload — Hilbert-space dimension, number of grid points, eigenvalue count, and
whether sparse diagonalization applies — and returns a recommended `num_cpus` and
BLAS-thread cap. It is a *pure* function (it starts no workers), so it is safe to call
anywhere. For the three-transmon system above (dressed dimension 6³ = 216), at the two grid
sizes from Demos 1 and 2:

In [ ]:
for n_points in (16, 384):
    cfg = scq.recommend_parallelization(dimension=6**3, num_points=n_points, evals_count=20)
    print(f"{n_points:>4} points -> num_cpus={cfg.num_cpus}, blas_threads={cfg.blas_threads}")
    print(f"            {cfg.reason}")


The recommendation reproduces what the demos showed: serial for the 16-point sweep,
parallel for the 384-point one. Two conveniences build on it:

- pass **`num_cpus="auto"`** to a sweep and it tunes itself *before* it runs:
  `scq.ParameterSweep(..., num_cpus="auto")`;
- set **`scq.settings.AUTO_PARALLEL = True`** to apply this to every sweep that does not
  specify `num_cpus`.

For a recommendation tuned to *your* hardware (rather than built-in defaults), run the
one-time **`scq.calibrate_parallelization()`** — it measures this machine's per-task
overhead, pool-startup cost, and per-point cost (writing
`~/.scqubits/parallel_calibration.json`), after which the recommendation uses your measured
break-even.

### Summary

- **Easiest:** let `recommend_parallelization` / `num_cpus="auto"` choose; optionally run
  `calibrate_parallelization()` once for machine-tuned advice.
- Parallelism helps only when *grid size × cost-per-point* greatly exceeds the per-task
  overhead; on small sweeps it does nothing, or slows you down — expected, not a bug
  (Demo 1 vs Demo 2).
- **Cap BLAS threads** to avoid oversubscription (the 90× example); keep
  `num_cpus × BLAS-threads ≈ cores`. Capping before import is the blunt manual route;
  `MULTIPROC_BLAS_THREADS` and the heuristic handle it per sweep.
- For large composite Hilbert spaces, **sparse diagonalization is usually a bigger lever
  than multiprocessing** — try it first.
- **Every number here is machine-specific** — measure (or calibrate) on your own hardware.

In [ ]:
# The settings touched in this notebook, with their current values:
print("NUM_CPUS               =", scq.settings.NUM_CPUS)
print("MULTIPROC_BLAS_THREADS =", scq.settings.MULTIPROC_BLAS_THREADS)
print("AUTO_SPARSE_DIAG       =", scq.settings.AUTO_SPARSE_DIAG)
print("MULTIPROC              =", scq.settings.MULTIPROC, "(parallelization backend)")
